> **Local runner:** The experiment code is in `scripts/`. Set the paths to your existing research inputs below. The notebook does not change the original data.

# Validate the saved clustering evidence

The optional diagnosis-masking run requires the original reconstructed inputs and fitted-model provenance.

In [ ]:
from pathlib import Path

def repository_root():
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "scripts" / "validate_experiment_evidence.py").is_file():
            return folder
    raise FileNotFoundError("Open Jupyter from inside the repository directory")

ROOT = repository_root()
PRIMARY_PROJECT = Path(r"C:\Projects\healthcare-cluster-reliability")  # Original primary research folder
RECON_PROJECT = Path(r"C:\Projects\JBHI_SectionV_Package")  # Folder with reconstruction run outputs
RAW_CSV = Path(r"C:\Projects\data\diabetic_data.csv")
SCRIPT = ROOT / "scripts" / "validate_experiment_evidence.py"
OUT = ROOT / "outputs" / "evidence_validation"
RUN_NEW_POSITION_MASKING = False
print("Validation script:", SCRIPT)
print("Reconstruction records:", RECON_PROJECT.is_dir())
print("Public UCI input:", RAW_CSV.is_file())

## 1. Existing-evidence checks

No clustering is rerun in this first pass. The script records each source file's SHA-256 and reports `NOT_EVALUATED` for unavailable evidence.

In [ ]:
import subprocess, sys
command = [sys.executable, str(SCRIPT), "--project", str(RECON_PROJECT), "--primary-project", str(PRIMARY_PROJECT), "--raw-csv", str(RAW_CSV), "--out", str(OUT)]
if RUN_NEW_POSITION_MASKING:
    command.append("--run-reconstruction-masking")
print("Starting validation; masking experiment enabled:", RUN_NEW_POSITION_MASKING)
r = subprocess.run(command, capture_output=True, text=True)
print("RETURN CODE:", r.returncode)
print(r.stdout)
if r.stderr:
    print("STDERR:\n", r.stderr)
if r.returncode != 0:
    raise RuntimeError("Validation failed; inspect STDERR and validation_evidence_register.csv")

## 2. Review evidence statuses

`PASS` establishes only the exact check stated in the `details` column. `NOT_EVALUATED` is never treated as a successful result.

In [ ]:
import pandas as pd
from IPython.display import display
register = pd.read_csv(OUT / "validation_evidence_register.csv")
display(register)
print("Status counts:", register.status.value_counts().to_dict())

## 3. Display measured results and their hashes

In [ ]:
import json
manifest = json.loads((OUT / "validation_manifest.json").read_text(encoding="utf-8"))
for name, result in manifest["results"].items():
    print("\n", name, "\n", json.dumps(result, indent=2))
print("\nSource input hashes:")
for name, metadata in manifest["source_files"].items():
    print(Path(name).name, metadata["sha256"])

## 4. Optional new experiment and figures

Set `RUN_NEW_POSITION_MASKING = True` in Cell 1 and run from Cell 1 onward. This checks *fixed-model* masking independently for positions 1, 2 and 3 at reconstructed weights 0.6 and 1.0. Do not report a result unless the new fit and saved-label checks pass.

In [ ]:
import matplotlib.pyplot as plt
mask_path = OUT / "NEW_reconstruction_position_masking_summary.csv"
if mask_path.is_file():
    s = pd.read_csv(mask_path)
    display(s)
    fig, ax = plt.subplots(figsize=(7,4))
    for weight, df in s.groupby("weight"):
        df = df.sort_values("diagnosis_position")
        ax.plot(df.diagnosis_position, df.mean_ari, marker="o", label=f"w={weight:g}")
    ax.set_xticks([1,2,3])
    ax.set_xlabel("Masked diagnosis position")
    ax.set_ylabel("Mean fixed-model ARI (20% masking)")
    ax.set_title("New reconstruction-only diagnosis-position masking")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("No new masking output yet. Expected on reference-only run.")

### Writing into manuscript

For Section V-B, use the 20-run mean, SD, median and range from the raw run-level CSV. For Section V-C, report the **primary** R1-versus-circulatory cross-tab only if the saved profile matches the primary development IDs. For Section V-D, report the reconstruction's new position-specific masking only after model replay matches the saved 06C selected fits and labels. Do not equate these two cohort splits or claim clinical validation.